In [19]:
import pandas as pd
import os

# Configurações

In [93]:
def carrega_df_socios(versao_base_cnpj):
    path_dataframes_base_cnpj = './bronze/base_cnpj/'+versao_base_cnpj+'/'
    # os.listdir(path_dataframes_base_cnpj)

    print(path_dataframes_base_cnpj)
    df = pd.read_parquet(path_dataframes_base_cnpj + 'df_socios.parquet')

    print(versao_base_cnpj,df.shape[0],df['dta_entrada_sociedade'].max())
    
    return df

In [92]:
444897
445970

'./bronze/base_cnpj/2025-10-12/'

In [83]:
df_202510 = carrega_df_socios('2025-10-12')

2025-10-12 25684773


In [91]:
df_202509 = carrega_df_socios('2025-09-14')

2025-09-14 25684773 2025-04-19


In [84]:
df_202504 = carrega_df_socios('2025-04-19')

2025-04-19 25684773


In [94]:
df_202305 = carrega_df_socios('2023-05-16')

./bronze/base_cnpj/2023-05-16/
2023-05-16 25684773 2025-04-19


In [99]:
df = pd.read_parquet('./bronze/base_cnpj/2023-05-16/socios/df_socios1.parquet')

In [100]:
df['dta_entrada_sociedade'].max()

datetime.date(2023, 6, 30)

In [75]:
cnpj_alvo = '00372383000129'

In [69]:
# df_202504 = df[df['cnpjbase'] == cnpj_alvo[0:8]].copy()
# df_202510 = df[df['cnpjbase'] == cnpj_alvo[0:8]].copy()

In [59]:
df_202504

,cnpjbase,tpsocio,nosocio,cnpj_cpf,cdqualisocio,dta_entrada_sociedade,pais,idrepre,noreprese,cdqualirepre,cdidade
23019890,00372383,2,GIOVANE IBERE CAVALCANTE DE FREITAS,***650421**,22,1993-12-02,None,***000000**,None,00,8
23019891,00372383,2,POLLYANNA VAZ CAVALCANTE PRUDENTE,***228841**,49,2014-10-06,None,***000000**,None,00,4
23019892,00372383,2,ALVARO SILVEIRA JUNIOR,***861641**,49,2016-08-24,None,***000000**,None,00,6
23019893,00372383,2,RODRIGO SILVEIRA,***403841**,22,2016-08-24,None,***000000**,None,00,6


In [23]:
def processa_lista(dicionario,lista_tipos):
    
    for tipo in lista_tipos:
        cronometro = datetime.now()

        print('Carrega arquivo: [{}]'.format(tipo))
        df = get_df_from_zip(tipo)
        mb_puro = df.memory_usage(deep=True).sum()/1024/1024

        print('Converte campos: [{}]'.format(tipo))
        dicionario[tipo] = converte_campos(tipo,df)
        mb_convertido = dicionario[tipo].memory_usage(deep=True).sum()/1024/1024

        print('Gera Arquivo Parquet: [{}]'.format(tipo))
        parquet_nome = exporta_df(tipo,dicionario[tipo])
        mb_parquet = os.stat(parquet_nome).st_size/1024/1024

        tempo = datetime.now() - cronometro
        print('Espaço {:.1f}/{:.1f}/{:.1f} mb Tempo total: {}'.format(mb_puro,mb_convertido,mb_parquet,tempo))

        # df_unicos[tipo].info(memory_usage='deep'),
        print()

In [ ]:
tipo = 'socios'
nome_parquet = path_dataframes_base_cnpj + tipo + '/df_' + tipo + '{}.parquet'

cronometro = datetime.now()

print('Carrega e concatena dataframes: [{}]'.format(tipo))

# Concatena Dataframes
df = pd.read_parquet(nome_parquet.format(0))
for digito in range(1,10):
    print('Carrega e concatena dataframes: {}'.format(digito))
    df = pd.concat(
        [df, pd.read_parquet(nome_parquet.format(digito))], 
        ignore_index=True)
    
tempo = datetime.now() - cronometro
print('Carga concluida: {}'.format(tempo))

# Converte Campos
df = converte_campos(tipo,df)

# Reinicia o index (aumenta o espaço necessário)
df.reset_index(inplace=True,drop=True)

# Exporta dataframe
exporta_df(tipo,df)

df.info(memory_usage='deep')

In [24]:
lista_arquivos

['K3241.K03200Y0.D50419.SOCIOCSV',
 'K3241.K03200Y1.D30513.SOCIOCSV',
 'K3241.K03200Y2.D30513.SOCIOCSV',
 'K3241.K03200Y3.D30513.SOCIOCSV',
 'K3241.K03200Y4.D30513.SOCIOCSV',
 'K3241.K03200Y5.D30513.SOCIOCSV',
 'K3241.K03200Y6.D30513.SOCIOCSV',
 'K3241.K03200Y7.D30513.SOCIOCSV',
 'K3241.K03200Y8.D30513.SOCIOCSV']

In [11]:
lista_df = []
for arquivo in lista_arquivos:
    df = pd.read_csv(path_arquivos + arquivo, sep=';', quotechar='"', encoding='ISO-8859-1')
    lista_df.append(df)

In [17]:
lista_df[0]

,00733116,2,MARCIO HENRIQUE MOREIRA,***042808**,49,20040920,Unnamed: 6,***000000**,Unnamed: 8,00,6
0,4057320,2,VIVIANE RAMOS,***389559**,49,20000922,NaN,***000000**,NaN,0,5
1,4057320,2,EMERSON GIOVANI DE SOUZA,***981009**,22,20000922,NaN,***000000**,NaN,0,6
2,4057336,2,IVANOR PEREIRA DOS SANTOS,***283771**,49,20000914,NaN,***000000**,NaN,0,7
3,4057336,2,SILVIA FLAVIA LOURENCO DA SILVA,***191498**,49,20031010,NaN,***000000**,NaN,0,5
4,2353244,2,GISELIA BISPO ALVES,***756318**,22,19980109,NaN,***000000**,NaN,0,8
...,...,...,...,...,...,...,...,...,...,...,...
7512417,60462706,2,PEDRO HENRIQUE VIEIRA DA COSTA,***335611**,49,20250419,NaN,***000000**,NaN,0,3
7512418,60462951,2,EFLAVIA GEOVANA BORGES DA SILVA,***033023**,49,20250419,NaN,***000000**,NaN,0,2
7512419,60463110,2,HENRIQUE DOUGLAS COSTA DA SILVA,***978792**,49,20250419,NaN,***000000**,NaN,0,3
7512420,60463164,2,EDIVALDO ALVES DO NASCIMENTO JUNIOR,***796335**,49,20250419,NaN,***000000**,NaN,0,4


In [12]:
df_socios = pd.concat(lista_df)
df_socios

,00733116,2,MARCIO HENRIQUE MOREIRA,***042808**,49,20040920,Unnamed: 6,***000000**,Unnamed: 8,00,...,20150902,01975412,REGIS ANTONIO MASSARELLI,***204909**,19970716,8,03168214,NIVIA MARIA DOS SANTOS,***214148**,19990517
0,4057320.0,2,VIVIANE RAMOS,***389559**,49.0,20000922.0,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4057320.0,2,EMERSON GIOVANI DE SOUZA,***981009**,22.0,20000922.0,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4057336.0,2,IVANOR PEREIRA DOS SANTOS,***283771**,49.0,20000914.0,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4057336.0,2,SILVIA FLAVIA LOURENCO DA SILVA,***191498**,49.0,20031010.0,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2353244.0,2,GISELIA BISPO ALVES,***756318**,22.0,19980109.0,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019144,NaN,2,NaN,NaN,49.0,NaN,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,5784359.0,LISIANE CARVALHO MIKALAUSCAS,***513150**,20221212.0
2019145,NaN,2,NaN,NaN,22.0,NaN,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,5784359.0,DANIEL DA ROSA OLIVEIRA,***827900**,20221212.0
2019146,NaN,2,NaN,NaN,49.0,NaN,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,2422920.0,MARIA DO CARMO CAPISTRANO GENOVEZ,***686899**,19980316.0
2019147,NaN,2,NaN,NaN,49.0,NaN,NaN,***000000**,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,2422920.0,FERNANDO GENOVEZ FILHO,***618979**,19980316.0
